# ***Cell 1 Imports & Configuration***

In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", font_scale=1.1)

# ***Cell 2 Mount Google Drive***

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# ***Cell 3 Dataset Path & Output Directory***

In [ ]:
RANDOM_SEED = 42
PIPELINE_VERSION = "v1.0.0"

np.random.seed(RANDOM_SEED)

DRIVE_ZIP = "/content/drive/MyDrive/cleaned_dataset.zip"
OUTPUT_DIR = "/content/processed_battery_dataset"

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError("cleaned_dataset.zip not found in Drive.")

print("Dataset:", DRIVE_ZIP)
print("Output:", OUTPUT_DIR)

Dataset: /content/drive/MyDrive/cleaned_dataset.zip
Output: /content/processed_battery_dataset


# ***Cell 4 Extract Dataset***

In [ ]:
DATASET_DIR = "/content/cleaned_dataset"

if not os.path.exists(DATASET_DIR):
    with zipfile.ZipFile(DRIVE_ZIP, "r") as zip_ref:
        zip_ref.extractall("/content")

print("Dataset extracted successfully.")

KeyboardInterrupt: 

# ***Cell 5 Locate Metadata***

In [ ]:
import glob

metadata_files = glob.glob(
    "/content/cleaned_dataset/**/metadata.csv",
    recursive=True
)

if not metadata_files:
    raise FileNotFoundError("metadata.csv not found.")

METADATA_PATH = metadata_files[0]
DATA_CSV_DIR = os.path.join(
    os.path.dirname(METADATA_PATH),
    "data"
)

print("Metadata:", METADATA_PATH)
print("Data directory:", DATA_CSV_DIR)

Metadata: /content/cleaned_dataset/metadata.csv
Data directory: /content/cleaned_dataset/data


# ***Cell 6 Load & Clean Metadata***

In [ ]:
df = pd.read_csv(METADATA_PATH)

df["Capacity"] = pd.to_numeric(
    df["Capacity"]
    .astype(str)
    .str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)

dis_df = (
    df[
        (df["type"] == "discharge") &
        (df["Capacity"] > 0.05)
    ]
    .sort_values(["battery_id", "test_id"])
    .copy()
)

print("Total records:", len(dis_df))
print("Total batteries:", dis_df["battery_id"].nunique())

display(dis_df.head())

Total records: 2714
Total batteries: 34


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
5121,discharge,[2.0080e+03 4.0000e+00 2.0000e+00 1.5000e+01 2...,24,B0005,1,5122,05122.csv,1.856487,NaN,NaN
5123,discharge,[2.0080e+03 4.0000e+00 2.0000e+00 1.9000e+01 4...,24,B0005,3,5124,05124.csv,1.846327,NaN,NaN
5125,discharge,[2.008e+03 4.000e+00 3.000e+00 0.000e+00 1.000...,24,B0005,5,5126,05126.csv,1.835349,NaN,NaN
5127,discharge,[2008. 4. 3. 4. 16. ...,24,B0005,7,5128,05128.csv,1.835263,NaN,NaN
5129,discharge,[2008. 4. 3. 8. 33. ...,24,B0005,9,5130,05130.csv,1.834646,NaN,NaN


In [ ]:
df.head(20)

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.674305,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.524366,NaN,NaN
5,charge,[2010. 7. 21. 22. 38. ...,4,B0047,5,6,00006.csv,NaN,NaN,NaN
6,discharge,[2.010e+03 7.000e+00 2.200e+01 1.000e+00 4.000...,4,B0047,6,7,00007.csv,1.508076,NaN,NaN
7,charge,[2010. 7. 22. 3. 14. ...,4,B0047,7,8,00008.csv,NaN,NaN,NaN
8,discharge,[2010. 7. 22. 6. 16. ...,4,B0047,8,9,00009.csv,1.483558,NaN,NaN
9,charge,[2010. 7. 22. 7. 50. ...,4,B0047,9,10,00010.csv,NaN,NaN,NaN


# ***Cell 7 Calculate Initial Capacity & SOH***

In [ ]:
initial_capacity = (
    dis_df
    .groupby("battery_id")["Capacity"]
    .first()
)

dis_df["initial_capacity"] = (
    dis_df["battery_id"]
    .map(initial_capacity)
)

dis_df["SOH"] = (
    dis_df["Capacity"] /
    dis_df["initial_capacity"]
)

print("SOH calculated successfully.")

display(
    dis_df[
        ["battery_id", "test_id", "Capacity", "initial_capacity", "SOH"]
    ].head()
)

# ***Cell 8 Calculate RUL***

In [ ]:
def assign_rul(group):

    eol_threshold = (
        0.70 * group["initial_capacity"].iloc[0]
    )

    eol_cycles = group.loc[
        group["Capacity"] <= eol_threshold,
        "test_id"
    ]

    if len(eol_cycles) > 0:
        eol_cycle = eol_cycles.iloc[0]
    else:
        eol_cycle = group["test_id"].max()

    group["EOL_cycle"] = eol_cycle

    group["RUL"] = np.maximum(
        0,
        eol_cycle - group["test_id"]
    )

    return group


dis_df = (
    dis_df
    .groupby("battery_id", group_keys=False)
    .apply(assign_rul)
)

print("RUL calculated successfully.")

display(
    dis_df[
        ["battery_id", "test_id", "Capacity", "SOH", "EOL_cycle", "RUL"]
    ].head()
)

# ***Cell 9 Define Time Grid & Channels***

In [ ]:
GRID_DT = 10
GRID_MAX_T = 500

TIME_GRID = np.arange(
    0,
    GRID_MAX_T + GRID_DT,
    GRID_DT
)

GRID_LEN = len(TIME_GRID)

CHANNELS = [
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured"
]

print("Time steps per cycle:", GRID_LEN)
print("Channels:", CHANNELS)

# ***Cell 10 Align All Discharge Cycles***

In [ ]:
aligned_cycles = []
meta_records = []

for _, row in dis_df.iterrows():

    file_path = os.path.join(
        DATA_CSV_DIR,
        row["filename"]
    )

    if not os.path.exists(file_path):
        continue

    try:
        raw = pd.read_csv(file_path)

        if (
            len(raw) < 10 or
            "Time" not in raw.columns or
            raw["Time"].iloc[-1] < 60
        ):
            continue

        raw = (
            raw
            .drop_duplicates(subset=["Time"])
            .sort_values("Time")
        )

        t_raw = raw["Time"].values

        cycle_tensor = np.zeros(
            (GRID_LEN, len(CHANNELS)),
            dtype=np.float32
        )

        valid = True

        for channel_idx, channel in enumerate(CHANNELS):

            if channel not in raw.columns:
                valid = False
                break

            interpolation = interp1d(
                t_raw,
                raw[channel].values,
                kind="linear",
                bounds_error=False,
                fill_value="extrapolate"
            )

            cycle_tensor[:, channel_idx] = (
                interpolation(TIME_GRID)
            )

        if not valid:
            continue

        aligned_cycles.append(cycle_tensor)

        meta_records.append({
            "battery_id": row["battery_id"],
            "test_id": int(row["test_id"]),
            "ambient_temperature": row["ambient_temperature"],
            "Capacity": float(row["Capacity"]),
            "SOH": float(row["SOH"]),
            "RUL": int(row["RUL"]),
            "filename": row["filename"]
        })

    except Exception:
        continue

aligned_X = np.stack(aligned_cycles, axis=0)
meta_df = pd.DataFrame(meta_records)

print("Aligned tensor shape:", aligned_X.shape)
print("Metadata shape:", meta_df.shape)

# ***Cell 11 Define Test Batteries***

In [ ]:
test_cells = [
    "B0018",
    "B0028",
    "B0032",
    "B0036",
    "B0040",
    "B0043",
    "B0044",
    "B0051",
    "B0056"
]

train_indices = meta_df[
    ~meta_df["battery_id"].isin(test_cells)
].index.values

print("Training samples:", len(train_indices))
print("Test batteries:", len(test_cells))
print("Test cells:", test_cells)

# ***Cell 12 Fit StandardScaler on Training Batteries Only***

In [ ]:
scaler = StandardScaler()

train_data = aligned_X[
    train_indices
].reshape(-1, len(CHANNELS))

scaler.fit(train_data)

print("Scaler fitted on training batteries only.")
print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)

# ***Cell 13 Scale Complete Dataset***

In [ ]:
aligned_X_scaled = np.zeros_like(aligned_X)

for i in range(len(aligned_X)):

    aligned_X_scaled[i] = scaler.transform(
        aligned_X[i]
    )

print("Scaling completed.")
print("Scaled tensor shape:", aligned_X_scaled.shape)

# ***Cell 14 Create Sliding Windows***

In [ ]:
WINDOW_SIZE = 10

windowed_X = []
windowed_y_soh = []
windowed_y_rul = []
windowed_y_cap = []
windowed_meta = []

for battery_id, group in meta_df.groupby("battery_id"):

    group_indices = group.index.values

    if len(group_indices) < WINDOW_SIZE:
        continue

    for start in range(
        len(group_indices) - WINDOW_SIZE + 1
    ):

        end = start + WINDOW_SIZE

        target_idx = group_indices[end - 1]

        window_tensor = aligned_X_scaled[
            group_indices[start:end]
        ]

        windowed_X.append(window_tensor)

        windowed_y_soh.append(
            meta_df.loc[target_idx, "SOH"]
        )

        windowed_y_rul.append(
            meta_df.loc[target_idx, "RUL"]
        )

        windowed_y_cap.append(
            meta_df.loc[target_idx, "Capacity"]
        )

        windowed_meta.append({
            "battery_id": battery_id,
            "target_test_id": int(
                meta_df.loc[target_idx, "test_id"]
            ),
            "ambient_temperature": meta_df.loc[
                target_idx,
                "ambient_temperature"
            ],
            "window_start_idx": start,
            "window_end_idx": end - 1
        })

windowed_X = np.stack(windowed_X)

windowed_y_soh = np.array(
    windowed_y_soh,
    dtype=np.float32
)

windowed_y_rul = np.array(
    windowed_y_rul,
    dtype=np.float32
)

windowed_y_cap = np.array(
    windowed_y_cap,
    dtype=np.float32
)

windowed_meta_df = pd.DataFrame(
    windowed_meta
)

print("Sliding-window shape:", windowed_X.shape)

# ***Cell 15 Save Aligned Cycle Dataset***

In [ ]:
aligned_path = os.path.join(
    OUTPUT_DIR,
    f"battery_aligned_cycles_{PIPELINE_VERSION}.npz"
)

np.savez_compressed(
    aligned_path,
    X=aligned_X_scaled,
    y_soh=meta_df["SOH"].values.astype(np.float32),
    y_rul=meta_df["RUL"].values.astype(np.float32),
    y_cap=meta_df["Capacity"].values.astype(np.float32)
)

print("Saved:", aligned_path)

# ***Cell 16 Save Sliding Window Dataset***

In [ ]:
window_path = os.path.join(
    OUTPUT_DIR,
    f"battery_sliding_windows_{PIPELINE_VERSION}.npz"
)

np.savez_compressed(
    window_path,
    X_windows=windowed_X,
    y_soh=windowed_y_soh,
    y_rul=windowed_y_rul,
    y_cap=windowed_y_cap
)

print("Saved:", window_path)

# ***Cell 17 Save Metadata***

In [ ]:
aligned_meta_path = os.path.join(
    OUTPUT_DIR,
    f"aligned_cycles_metadata_{PIPELINE_VERSION}.csv"
)

window_meta_path = os.path.join(
    OUTPUT_DIR,
    f"sliding_windows_metadata_{PIPELINE_VERSION}.csv"
)

meta_df.to_csv(
    aligned_meta_path,
    index=False
)

windowed_meta_df.to_csv(
    window_meta_path,
    index=False
)

print("Metadata files saved.")

# ***Cell 18 Save Pipeline Configuration***

In [ ]:
config = {
    "version": PIPELINE_VERSION,
    "seed": RANDOM_SEED,

    "grid_dt_seconds": GRID_DT,
    "grid_max_time_seconds": GRID_MAX_T,
    "time_steps_per_cycle": GRID_LEN,

    "channels": CHANNELS,

    "scaling": "StandardScaler_fit_on_train_cells_only",

    "window_size_cycles": WINDOW_SIZE,

    "total_aligned_cycles": int(
        len(aligned_X)
    ),

    "total_sliding_windows": int(
        len(windowed_X)
    ),

    "aligned_tensor_shape": list(
        aligned_X.shape
    ),

    "windowed_tensor_shape": list(
        windowed_X.shape
    ),

    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist()
}

config_path = os.path.join(
    OUTPUT_DIR,
    f"pipeline_config_{PIPELINE_VERSION}.json"
)

with open(config_path, "w") as file:
    json.dump(
        config,
        file,
        indent=4
    )

print("Configuration saved:")
print(config_path)

# ***Cell 19 Final Verification***

In [ ]:
print("=" * 60)
print("PREPROCESSING PIPELINE COMPLETE")
print("=" * 60)

print(f"Aligned cycles      : {len(aligned_X)}")
print(f"Aligned shape       : {aligned_X.shape}")

print(f"Sliding windows     : {len(windowed_X)}")
print(f"Window shape        : {windowed_X.shape}")

print(f"Training samples    : {len(train_indices)}")
print(f"Test batteries      : {len(test_cells)}")

print("\nSaved files:")

for file in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", file)

# ***Cell 20 Battery Cycle Visualization***

In [ ]:
battery_id = "B0005"

battery_meta = meta_df[
    meta_df["battery_id"] == battery_id
].sort_values("test_id")

selected_indices = [
    battery_meta.index[0],
    battery_meta.index[len(battery_meta) // 2],
    battery_meta.index[-1]
]

labels = ["Early Cycle", "Middle Cycle", "Late Cycle"]

fig, axes = plt.subplots(3, 1, figsize=(12, 12))

for idx, label in zip(selected_indices, labels):

    cycle = aligned_X[idx]

    axes[0].plot(
        TIME_GRID,
        cycle[:, 0],
        label=label
    )

    axes[1].plot(
        TIME_GRID,
        cycle[:, 1],
        label=label
    )

    axes[2].plot(
        TIME_GRID,
        cycle[:, 2],
        label=label
    )

axes[0].set_title(f"Voltage Profile — {battery_id}")
axes[0].set_ylabel("Voltage (V)")
axes[0].legend()

axes[1].set_title(f"Current Profile — {battery_id}")
axes[1].set_ylabel("Current (A)")
axes[1].legend()

axes[2].set_title(f"Temperature Profile — {battery_id}")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Temperature (°C)")
axes[2].legend()

plt.tight_layout()
plt.show()

# ***Cell 21 SOH & RUL Visualization***

In [ ]:
battery_id = "B0005"

battery_data = meta_df[
    meta_df["battery_id"] == battery_id
].sort_values("test_id")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    battery_data["test_id"],
    battery_data["SOH"],
    marker="o"
)

axes[0].axhline(
    0.70,
    linestyle="--",
    label="70% EOL"
)

axes[0].set_title(f"SOH Degradation — {battery_id}")
axes[0].set_xlabel("Cycle Number")
axes[0].set_ylabel("SOH")
axes[0].legend()

axes[1].plot(
    battery_data["test_id"],
    battery_data["RUL"],
    marker="o"
)

axes[1].set_title(f"RUL Degradation — {battery_id}")
axes[1].set_xlabel("Cycle Number")
axes[1].set_ylabel("RUL (Cycles)")

plt.tight_layout()
plt.show()

# ***Cell 22 Dataset Tensor Visualization***

In [ ]:
print("PREPROCESSED DATA")

print("Aligned X shape :", aligned_X_scaled.shape)
print("Window X shape  :", windowed_X.shape)

print("\nTargets:")
print("SOH shape       :", windowed_y_soh.shape)
print("RUL shape       :", windowed_y_rul.shape)
print("Capacity shape  :", windowed_y_cap.shape)